In [1]:
import xarray as xr
import pandas as pd
import glob
import os
import math

import numpy as np
import re
from joblib import Parallel, delayed
from pathlib import Path
import matplotlib.pyplot as plt
import alphashape
from itertools import combinations
from collections import defaultdict, deque
from concurrent.futures import ThreadPoolExecutor

from concurrent.futures import ProcessPoolExecutor, as_completed


from shapely import points, contains
import random

In [2]:
working_dir = '/glade/work/qingyuany/repo_data/'
case_name = "test"

In [3]:
def meta_one_hot_shot(meta, para_nm):
    meta = meta.transpose()
    meta_one_hot = pd.DataFrame(False, index=meta.index, columns=para_nm)
    for index, row in meta.iterrows():
        for r in row.values:
            meta_one_hot.loc[index, para_nm[r]] = True

    return meta_one_hot

In [4]:
class EmulatedDataStorage:
    """
    Lightweight container for emulation outputs.
    No computation logic.
    """
    def __init__(self):
        pass

In [5]:
def sample_from_hull(X, para, h):

    minx, miny, maxx, maxy = h.bounds
    p1, p2 = para

    X = X[
        (X[p1] >= minx) & (X[p1] <= maxx) &
        (X[p2] >= miny) & (X[p2] <= maxy)
    ]

    if X.empty:
        return X
    
    pts = points(X[list(para)].values)
    tf = contains(h,pts)
    
    return X.loc[tf].reset_index(drop=True)

In [6]:
def _one_batch(args):
    para_l, para_nm, grouped_hulls, n_pts = args

    X = pd.DataFrame(
        np.random.uniform(0, 1, (n_pts, len(para_nm))),
        columns=para_nm
    )

    for para in para_l:
        h = grouped_hulls[para]
        X = sample_from_hull(X, para, h)

        if X.shape[0] == 0:
            return None

    return X



In [7]:
def test_ind_vars(X_prev, X, para_nm, tf_masks, grouped_hulls, para, paras_vars, shape_alpha = 5):
    print(f'\t \tRunning test to see if {para} could be break down and lead to non-overlapping')

    vars = paras_vars[para]
    for i in reversed(range(len(vars))):
        var_combs = combinations(vars, i)
        for var_comb in var_combs:
            
            X_sub = X[tf_masks[list(var_comb)].all(axis = 1)]
            if X_sub.shape[0] > 5000:
                X_sub = X_sub.sample(5000)
            
            X_sub = X_sub[list(para)].values
            hull_sub = alphashape.alphashape(X_sub, shape_alpha)

            attempt = sample_from_hull(X_prev, para, hull_sub)
            
            if not attempt.empty:
                print(f'\t \t \t Found the good variable combo')
                drop_vars = [x for x in vars if x not in list(var_comb)]
                return list(var_comb), drop_vars, hull_sub
            else:
                pass 

        if i -1 == 0:
            print(f'\t \t \t No variable combinations work')
            return None

In [45]:
def sample_from_hulls_n(
    para_l,
    para_nm,
    grouped_hulls,
    n_pts=100_000,
    n_threshold=100_000,
    max_workers=None,
):
    if max_workers is None:
        max_workers = os.cpu_count()

    out = []
    count = 0

    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        futures = []
        MAX_IN_FLIGHT = 2 * max_workers

        while count < n_threshold or futures:
            while count < n_threshold and len(futures) < MAX_IN_FLIGHT:
                futures.append(ex.submit(_one_batch,
                    (para_l, para_nm, grouped_hulls, n_pts),))
                
            # harvest finished jobs
                for fut in as_completed(list(futures)):
                    futures.remove(fut)
                    X = fut.result()
    
                    # if X is None:
                    #     ex.shutdown(cancel_futures=True)
                    #     return None
    
                    out.append(X)
                    if X is not None:
                        count += X.shape[0]
    
                    if count >= n_threshold:
                        ex.shutdown(cancel_futures=True)
                        break


                if all([x is None for x in out]):
                    return None
    
    out = [x for x in out if x is not None]
    out = pd.concat(out, axis=0).reset_index(drop = True)
    return out



In [46]:
def orchestrate_test(para_seq, X, tf_masks, para_nm, grouped_hulls, paras_vars, n_pts=10000, n_threshold=10000, max_workers=None):
    para_l = []
    para_non_over = []
    var_drop = {}
    
    non_over_count = 0
    for p in para_seq:
        print(f'Running {p}')
        para_l.append(p)
        out = sample_from_hulls_n(para_l, para_nm, grouped_hulls, n_pts=  n_pts, n_threshold = n_threshold, max_workers = 2)
        
        if out is None:
        
            check_pt = test_ind_vars(out_prev, X, para_nm, tf_masks, grouped_hulls, p, paras_vars, shape_alpha = 5)
            if check_pt is None:
                para_l.remove(p)
                print(f'\t \t \t \t {p} is causing trouble and is skipped')
                non_over_count = non_over_count + 1
                para_non_over.append(p)
                
                
            else:
                print(f'\t \t \t \t Modify the original hull and variable')
                print(f'\t \t \t \t Drop {check_pt[1]}')
                grouped_hulls[p] = check_pt[2]
                paras_vars[p] = check_pt[0]
                var_drop[p] = check_pt[1]
                out_prev = sample_from_hulls_n(para_l, para_nm, grouped_hulls, n_pts=  n_pts, n_threshold = n_threshold, max_workers = 2)
                
                
        else:
            out_prev  = out
            
    
    
        # if non_over_count > 5:
        #     print("Too many non-overlapping, break the loop")
        #     return para_l, para_non_over

        
        
    return para_l, para_non_over, grouped_hulls, paras_vars, var_drop

In [47]:
para_seq = list(test_case.grouped_hulls.keys())[:5]
check = orchestrate_test(para_seq, test_case.p_emu, test_case.tf_masks,  test_case.para_nm, test_case.grouped_hulls, test_case.paras_vars, n_pts=  5000, n_threshold = 5000, max_workers = 31)

Running ('clubb_C8', 'microp_aero_wsub_min')
Running ('clubb_C8', 'microp_aero_wsubi_scale')
Running ('dust_emis_fact', 'microp_aero_wsubi_scale')
Running ('clubb_c14', 'microp_aero_wsubi_scale')
Running ('clubb_c2rt', 'zmconv_dmpdz')


In [49]:
pts = sample_from_hulls_n(list(check[2].keys()), test_case.para_nm, check[2],n_pts=100000, n_threshold=100,max_workers=32)


In [50]:
for _ in range(100):
    pts = sample_from_hulls_n(list(check[2].keys()), test_case.para_nm, check[2],n_pts=100000, n_threshold=100000,max_workers=32)
    if pts is not None:
        print('find something')

In [21]:
# def sample_from_hull_above_n(para_l, para_nm, grouped_hulls, n_pts = 100000, n_threshold = 100000):
#     out = []
#     count = 0
#     while count < n_threshold:
#         X = pd.DataFrame(np.random.uniform(0, 1, (n_pts, len(para_nm))), columns = para_nm)
#         for para in para_l:
#             h = grouped_hulls[para]    
#             X = sample_from_hull(X, para, h)
            
#             if X.shape[0] == 0:
#                 print(f'Break by {para}')
#                 return None               
            
#         out.append(X)
#         count = count + X.shape[0]
                
#     out = pd.concat(out, axis = 0)
#     return out

In [10]:
def sample_from_hulls(X, paras, grouped_hulls):
    X = X
    for p in paras:
        h = grouped_hulls[p]
        X = sample_from_hull(X, p, h)
        if X.shape[0] == 0:
            print(f'Break by {para}')
            return None
            
    return X, X.shape[0]

In [11]:
class HistoryMatching:
    def __init__(self, working_dir, case_name, threshold_level = 2.0):
        self.root = Path(working_dir) / case_name
        self.tf_masks = pd.read_csv(self.root / f'tf_masks_level_{threshold_level}.csv', index_col=0)
        self.meta = pd.read_csv(self.root / 'meta.csv', index_col = 0)
        self.p_emu = xr.open_dataset(self.root / 'sampled_parameters.nc').to_dataframe()
        self.var_nm = list(self.tf_masks.columns)
        self.para_nm = list(self.p_emu.columns)

        self.tf_masks_raw = self.tf_masks

        self.dropped_vars = EmulatedDataStorage()
        self.n_sample = self.tf_masks.shape[0]

        self.dropped_vars.nooverlap2d = []
    def drop_by_name(self, var_to_exclude):
        var_to_drop = []
        for v in var_to_exclude:
            var_to_drop.extend([s for s in self.var_nm if s.startswith(v)])
        
        self.tf_masks = self.tf_masks.drop(columns = var_to_drop)

        self.var_nm = list(self.tf_masks.columns)
        self.dropped_vars.by_name = var_to_drop

    def drop_by_n_survive(self, n_survive):
        survive_summary = self.tf_masks.sum(axis = 0)
        self.dropped_vars.useless = list(survive_summary[survive_summary == self.n_sample].index)
        self.dropped_vars.tight   = list(survive_summary[survive_summary < n_survive].index)

        self.tf_masks = self.tf_masks.drop(columns = self.dropped_vars.useless + self.dropped_vars.tight)
        
        self.var_nm = list(self.tf_masks.columns)

    def update_meta(self, occurence_threshold = 2):
        self.meta = self.meta[self.var_nm]
        self.meta_onehot = meta_one_hot_shot(self.meta, self.para_nm)
        p_occur_count = self.meta_onehot.sum(axis = 0)
        self.p_occur_count = p_occur_count
        
#        p_sensitive = list(p_occur_count[p_occur_count > occurence_threshold].index)
#        self.meta_onehot = self.meta_onehot[p_sensitive]

    def hull_for_each(self, shape_alpha = 5):
        hull_per_var = {}
        for v in self.var_nm:
            p_ind = self.meta[v].sort_values().values
            pts = self.p_emu[self.tf_masks[v]].iloc[:,p_ind]
            if pts.shape[0] > 5000:
               pts = pts.sample(5000)

            pts = pts.values
            
            hull_per_var[v] = alphashape.alphashape(pts, shape_alpha)

        self.hull_per_var = hull_per_var

    def group_para_climatology(self):

        vars = self.var_nm
        paras_vars = {}
        paras_vars_0 = {}    
        for c in vars: 
            para_inds = self.meta[c].sort_values().values
            para_nms = [self.para_nm[ind] for ind in para_inds]
            paras_vars.setdefault(tuple(para_nms), []).append(c)



        paras_vars = dict(
                sorted(paras_vars.items(), key=lambda item: len(item[1]), reverse=True)
        )
        
        self.paras_vars = paras_vars
        
        
        for k, v in paras_vars.items():
            temp_count = self.tf_masks[v].all(axis = 1).sum()
            print(f'{k[0]:<40} and {k[1]:<40}: {temp_count:>8}')
            if temp_count < 10000:
                paras_vars_0[k] = v

        self.paras_vars_0 = paras_vars_0

        
        
    def shuffle_vars(self, n_comb = 2):
        summary_table = {}
    
        for paras, vars in self.paras_vars_0.items():
            
            vars_temp = vars 
            pd_list = []
            
            for vars_comb in combinations(vars_temp, n_comb):
                pd_list.append(list(vars_comb) + [self.tf_masks[list(vars_comb)].all(axis = 1).sum()])
                
            pd_list = pd.DataFrame(pd_list, columns = [f"var{i+1}" for i in range(n_comb)] + ["count"])
            pd_list = pd_list.sort_values(by = "count")
            
            
            summary_table[paras] = pd_list

        print(f'There are {len(self.paras_vars_0)} groups that have no overlapping within own groups')
        return summary_table

    def drop_no_overlap2d_vars(self, vars_to_drop):
        self.tf_masks = self.tf_masks.drop(columns = vars_to_drop)
        self.var_nm = list(self.tf_masks.columns)
        self.meta = self.meta[self.var_nm]
        self.meta_onehot = meta_one_hot_shot(self.meta, self.para_nm)
        self.dropped_vars.nooverlap2d.append(vars_to_drop)

        
    
    def visualize(self, para_pair):
        para_pair = tuple(para_pair)
        survive_pts = self.p_emu[self.tf_masks[self.paras_vars[para_pair]].all(axis = 1)]

        if survive_pts.shape[0] > 5000:
            survive_pts = survive_pts.sample(5000)

        return survive_pts

    def build_hulls(self, shape_alpha = 5):
        grouped_hulls = {}

        for para2, vars in self.paras_vars.items():

            tf_mask = self.tf_masks[vars].all(axis = 1)
            pts = self.p_emu[tf_mask]
            if pts.shape[0] > 5000:
                pts = pts.sample(5000)
            
            pts_np = pts[list(para2)].values
            
            grouped_hulls[para2] = alphashape.alphashape(pts_np, shape_alpha)
            
            
        self.grouped_hulls = grouped_hulls
    
    def sample_short_chain(self, n_pts = 100000, n_comb = 2):
        para_tuple = list(combinations(list(self.paras_vars.keys()),n_comb))
        r_pts = pd.DataFrame(np.random.uniform(0, 1, (n_pts, len(self.para_nm))), columns = self.para_nm)    
        
        def _worker(p2s):
            return p2s, sample_from_hulls(r_pts, p2s, self.grouped_hulls)[1]
        
        out = {}
        with ThreadPoolExecutor(max_workers=31) as ex:
            for k, v in ex.map(_worker, para_tuple):
                out[k] = v


        out = sorted(out.items(), key=lambda x: x[1])

        return out



    

In [12]:
test_case = HistoryMatching(working_dir, case_name)
test_case.drop_by_name(["TGCLDLWP", "CLDTOT"])
test_case.drop_by_n_survive(n_survive = 100000)
test_case.update_meta()

In [13]:
test_case.group_para_climatology()
summary2d = test_case.shuffle_vars()

clubb_C8                                 and microp_aero_wsub_min                    :   314967
clubb_C8                                 and microp_aero_wsubi_scale                 :    48436
dust_emis_fact                           and microp_aero_wsubi_scale                 :        0
clubb_c14                                and microp_aero_wsubi_scale                 :   176779
micro_mg_berg_eff_factor                 and microp_aero_wsubi_scale                 :        0
clubb_c2rt                               and zmconv_dmpdz                            :   869194
clubb_c2rt                               and microp_aero_wsubi_scale                 :        0
clubb_C8                                 and zmconv_capelmt                          :        0
clubb_C8                                 and cldfrc_dp1                              :   684794
zmconv_capelmt                           and seasalt_emis_scale                      :   548406
clubb_c14                               

In [14]:
no_overlap_2d_vars = ["FLUT_zonal_55to60", 'LWCF_zonal_-70to-65', 'FLUT_zonal_-60to-55', 
                      'FLUT_zonal_-55to-50', 'FLUT_zonal_-65to-60', 'SWCF_zonal_-60to-55', 
                      'SWCF_zonal_-55to-50', 'LWCF_zonal_70to75', 'LWCF_zonal_65to70', 'LWCF_zonal_30to35']

In [15]:
test_case.drop_no_overlap2d_vars(no_overlap_2d_vars)
test_case.group_para_climatology()
summary2d = test_case.shuffle_vars()

clubb_C8                                 and microp_aero_wsub_min                    :   314967
clubb_C8                                 and microp_aero_wsubi_scale                 :    48436
dust_emis_fact                           and microp_aero_wsubi_scale                 :    70638
clubb_c14                                and microp_aero_wsubi_scale                 :   176779
clubb_c2rt                               and zmconv_dmpdz                            :   869194
clubb_C8                                 and cldfrc_dp1                              :   684794
micro_mg_berg_eff_factor                 and microp_aero_wsubi_scale                 :   813089
zmconv_capelmt                           and seasalt_emis_scale                      :   548406
clubb_c14                                and clubb_c2rt                              :   205458
clubb_c2rt                               and microp_aero_wsubi_scale                 :   126587
micro_mg_berg_eff_factor                

In [16]:
test_case.build_hulls()
